# Fairness methodologies

This notebook shows simple usage of the fairness algorithms implemented.

In [1]:
from credit_pipeline.data import load_dataset
from credit_pipeline import training
from credit_pipeline.evaluate import get_fairness_metrics
from credit_pipeline.fairness_models import Reweighing, FairGBM, ThresholdOpt
from lightgbm import LGBMClassifier

/home/hiaac/miniconda3/envs/credit_pipeline/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading data and fitting a baseline model

We will use the german dataset with the gender as the sensitive attribute. The baseline algorithm and the fairness algrithms that employ a base estimator will use LGBM with default hyperparameters.

In [2]:
dataset_name = "german"
seed = 0
df = load_dataset(dataset_name)
train, test = training.create_train_test(df, final=True, seed=seed)
X_train = train.drop(columns=["DEFAULT"])
Y_train = train["DEFAULT"]
X_test = test.drop(columns=["DEFAULT"])
Y_test = test["DEFAULT"]

if dataset_name == "homecredit":
    A_train = X_train.CODE_GENDER == "M"
    A_test = X_test.CODE_GENDER == "M"
elif dataset_name == "taiwan":
    A_train = X_train.SEX == "Male"
    A_test = X_test.SEX == "Male"
elif dataset_name == "german":
    A_train = X_train.Gender == "Male"
    A_test = X_test.Gender == "Male"
A_train = A_train.astype(int)
A_test = A_test.astype(int)

In [3]:
# fit a baseline model
pipeline = training.create_pipeline(X_train, Y_train, classifier = LGBMClassifier(verbose = -1))
pipeline.fit(X_train, Y_train);

In [4]:
models_dict = {}
models_dict["baseline"] = pipeline.predict(X_test)

## Pre-processing

In [5]:
rw_model = Reweighing(LGBMClassifier(verbose = -1))
pipeline = training.create_pipeline(X_train, Y_train, classifier = rw_model)
pipeline.fit(X_train, Y_train, classifier__sensitive_attributes=A_train);
models_dict["reweighing"] = pipeline.predict(X_test)

## In-processing

In [6]:
fairgbm_model = FairGBM()
pipeline = training.create_pipeline(X_train, Y_train, classifier = fairgbm_model)
pipeline.fit(X_train, Y_train, classifier__sensitive_attributes=A_train);
models_dict["fairgbm"] = pipeline.predict(X_test)

## Post-processing

In [7]:
post_model = ThresholdOpt(LGBMClassifier(verbose = -1), constraint_value=0.05)
pipeline = training.create_pipeline(X_train, Y_train, classifier = post_model)
pipeline.fit(X_train, Y_train, classifier__sensitive_attributes=A_train);
models_dict["threshold_opt"] = pipeline.predict(X_test, sensitive_attributes=A_test)

## Evaluating models

In [8]:
get_fairness_metrics(
    models_dict,
    X_test,
    Y_test,
    A_test,
)

,model,DPD,EOD,AOD,APVD,GMA,balanced_accuracy
0,baseline,0.034352,0.227669,0.066924,0.206687,0.712852,0.661590
1,reweighing,-0.049933,-0.143791,-0.096334,0.102134,0.749269,0.681297
2,fairgbm,-0.028831,-0.044662,-0.052387,0.118995,0.732042,0.664046
3,threshold_opt,-0.008343,-0.017429,-0.033855,0.091460,0.710957,0.696746


The baseline model presented 0.22 of equality of opportunity difference, which is a very high value. This was reduced to 0.14 from the reweighing approach and both FairGBM and Threshold optimization were able to reach values below 0.05. These improvements could be made without any loss in balanced accuracy. Actually, threshold optimization had 0.69 of balanced accuracy, the highest value.